# Reproduce: Diff-RTPGHI Paper Results

This notebook reproduces **Table I** and **Figure 1** from:

> C. Hollomey, *Differentiable Real-Time Phase Reconstruction for Non-Uniform Filterbanks*, IEEE Signal Processing Letters, 2026.

It compares phase retrieval methods on a 134-channel auditory (ERB-scale) filterbank with half-ERB spacing and ~77x redundancy, using 10 synthetic test signals.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    'cool-frames @ git+https://github.com/allthatsounds/cool-frames.git'],
                   check=True)

import heapq
import math

import matplotlib.pyplot as plt

import numpy as np

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['figure.dpi'] = 100


In [ ]:
from cool_frames.numpy.filterbanks._core import filterbank, ifilterbank
from cool_frames.numpy.filterbanks._frame import filterbankrealdual
from cool_frames.numpy.filterbanks._utils import normalise_a
from cool_frames.numpy.filters._design import audfilters
from cool_frames.numpy.filters._gabfilters import _comp_tfrfromwin
from cool_frames.numpy.phase._admm import admm, dm, raar
from cool_frames.numpy.phase._diff_constphase import (
    constphase_nonuniform,
)
from cool_frames.numpy.phase._gla import gla
from cool_frames.numpy.phase._phasegrad import filterbankphasegrad

print('All imports OK.')


## Helper Functions

In [ ]:
def sdr_fn(ref, est):
    """Signal-to-distortion ratio in dB."""
    L = min(len(ref), len(est))
    r, e = ref[:L], est[:L]
    noise = r - e
    return 10 * np.log10(np.sum(r**2) / (np.sum(noise**2) + 1e-30))


def sc_roundtrip(c_target, c_recon, g, a_norm, L):
    """Round-trip spectral convergence: synthesise, re-analyse, compare magnitudes."""
    gd = filterbankrealdual(g, a_norm, L)
    sig = ifilterbank(c_recon, gd, a_norm, Ls=L, real=True)
    sig = np.real(sig)
    c_re = filterbank(sig, g, a_norm, L=L)
    r = np.concatenate([np.abs(np.asarray(ci).ravel()) for ci in c_target])
    e = np.concatenate([np.abs(np.asarray(ci).ravel()) for ci in c_re])
    return 20 * np.log10(np.linalg.norm(r - e) / (np.linalg.norm(r) + 1e-30))


def align_phase_sdr(sig_ref, c_recon, gd, a_norm, L, Ls, n_angles=720):
    """SDR with optimal global phase alignment (grid search)."""
    best_sdr = -200
    for theta in np.linspace(0, 2*np.pi, n_angles, endpoint=False):
        c_adj = [ci * np.exp(1j * theta) for ci in c_recon]
        sig_adj = ifilterbank(c_adj, gd, a_norm, Ls=L, real=True)
        sig_adj = np.real(sig_adj[:Ls])
        s = sdr_fn(sig_ref[:Ls], sig_adj)
        if s > best_sdr:
            best_sdr = s
    return best_sdr


def get_fc_norm(g, L):
    """Compute normalised centre frequencies from filter frequency responses."""
    M = len(g)
    fc_norm = np.zeros(M)
    for m in range(M):
        gm = g[m]
        if 'H' in gm:
            H_vals = np.asarray(gm['H'](L))
            fo = int(gm['foff'](L)) if callable(gm['foff']) else int(gm['foff'])
            k_abs = (np.arange(fo, fo + len(H_vals)) % L).astype(float)
            weights = np.abs(H_vals) ** 2
            ws = weights.sum()
            if ws > 0:
                k_cent = k_abs.copy()
                k_cent[k_cent > L / 2] -= L
                fc_norm[m] = np.sum(k_cent * weights) / ws / L * 2
    return fc_norm


def compute_tfr(g, L):
    """Compute TFR (time-frequency ratio) per channel."""
    M = len(g)
    tfr = np.zeros(M)
    for m in range(M):
        gm = g[m]
        if 'H' in gm:
            H_vals = np.asarray(gm['H'](L))
            h_time = np.fft.ifft(H_vals)
            h_real = np.real(np.fft.fftshift(h_time))
            gamma = _comp_tfrfromwin(h_real)
            tfr[m] = gamma / L if L > 0 else 0.0
    return tfr

## Batch Heap PGHI (Derivative-Filter Gradients)

In [ ]:
def batch_heap_pghi(sig_data, g, a_norm, L, a_int, fc_norm):
    """Batch heap-based PGHI using derivative-filter gradients."""
    M = len(g)
    tgrad_l, fgrad_l, s_l, c = filterbankphasegrad(sig_data, g, a_norm, L)
    c_flat = [np.asarray(ci).ravel() for ci in c]
    N = [len(ci) for ci in c_flat]
    Nsum = sum(N)
    offsets = np.zeros(M + 1, dtype=int)
    for m in range(M):
        offsets[m + 1] = offsets[m] + N[m]

    abss = np.concatenate([np.abs(ci) for ci in c_flat])
    tgw = np.concatenate([np.asarray(tg).ravel() for tg in tgrad_l]) * math.pi
    fgw = np.concatenate([-np.asarray(fg).ravel() for fg in fgrad_l]) * math.pi

    phase = np.zeros(Nsum)
    visited = np.zeros(Nsum, dtype=bool)

    def flat_idx(m, n): return offsets[m] + n % N[m]
    def get_mn(flat):
        m = int(np.searchsorted(offsets, flat + 1) - 1)
        return m, flat - offsets[m]

    seed = int(np.argmax(abss))
    visited[seed] = True
    heap = [(-abss[seed], seed)]

    while heap:
        _, flat = heapq.heappop(heap)
        m, n = get_mn(flat)
        for dn in [1, -1]:
            nn = n + dn
            if 0 <= nn < N[m]:
                fi = flat_idx(m, nn)
                if not visited[fi]:
                    phase[fi] = phase[flat] + dn * a_int[m] * (tgw[flat] + tgw[fi]) / 2
                    visited[fi] = True
                    heapq.heappush(heap, (-abss[fi], fi))
        for dm in [1, -1]:
            mm = m + dm
            if 0 <= mm < M:
                n_fn = int(round(n * a_int[m] / a_int[mm])) % N[mm]
                fi = flat_idx(mm, n_fn)
                if not visited[fi]:
                    dt = n_fn * a_int[mm] - n * a_int[m]
                    df = fc_norm[mm] - fc_norm[m]
                    if dm > 0 and df < 0: df += 2.0
                    if dm < 0 and df > 0: df -= 2.0
                    phase[fi] = (phase[flat] + dt * (tgw[flat] + tgw[fi]) / 2
                                 + df * (fgw[flat] + fgw[fi]) / 2)
                    visited[fi] = True
                    heapq.heappush(heap, (-abss[fi], fi))

    phase[~visited] = 0.0
    c_recon = []
    offset = 0
    for m in range(M):
        nm = N[m]
        c_recon.append(np.abs(c_flat[m]) * np.exp(1j * phase[offset:offset+nm]))
        offset += nm
    return c_recon, c

## Streaming Heap PGHI (Magnitude-Based Gradients)

In [ ]:
def _heap_phase_tick(slog_prev, slog_curr, tgradw_prev, tgradw_curr,
                     fgradw_curr, fc, prev_phase, tol, M,
                     time_prev, time_curr):
    """Heap-based phase integration for one streaming tick."""
    phase = np.zeros(M)
    visited = np.zeros(M, dtype=bool)
    all_slog = np.concatenate([slog_prev, slog_curr])
    logabstol = np.max(all_slog) + math.log(tol + np.finfo(float).tiny)
    below_tol = slog_curr <= logabstol

    heap = []
    priorities = np.concatenate([slog_prev, slog_curr])
    for w in range(2 * M):
        heapq.heappush(heap, (-priorities[w], w))

    while heap:
        neg_p, w = heapq.heappop(heap)
        if w >= M:
            m = w - M
            if below_tol[m] or not visited[m]:
                continue
            for dm, sign in [(1, 1), (-1, -1)]:
                mm = m + dm
                if 0 <= mm < M and not visited[mm] and not below_tol[mm]:
                    step_fc = fc[mm] - fc[m]
                    if dm > 0 and step_fc < 0: step_fc += 2.0
                    if dm < 0 and step_fc > 0: step_fc -= 2.0
                    dt = time_curr[mm] - time_curr[m]
                    phase[mm] = (phase[m]
                                 + dt * (tgradw_curr[m] + tgradw_curr[mm]) / 2.0
                                 + step_fc * (fgradw_curr[m] + fgradw_curr[mm]) / 2.0)
                    visited[mm] = True
        else:
            m = w
            if below_tol[m] or visited[m]:
                continue
            dt = time_curr[m] - time_prev[m]
            phase[m] = prev_phase[m] + dt * (tgradw_prev[m] + tgradw_curr[m]) / 2.0
            visited[m] = True

    rng = np.random.default_rng()
    for m in range(M):
        if not visited[m]:
            phase[m] = rng.uniform(0, 2 * np.pi)
    return phase


def streaming_heap_pghi(s_list, a, fc, sqtfr, L, tol=1e-6):
    """Full streaming heap PGHI with magnitude-based gradients."""
    a = np.asarray(a, dtype=int).ravel()
    fc = np.asarray(fc, dtype=float).ravel()
    M = len(a)
    N = np.array([len(s) for s in s_list], dtype=int)

    events = []
    for m in range(M):
        for n in range(N[m]):
            events.append((n * a[m], m, n))
    events.sort(key=lambda x: (x[0], x[1]))

    log_bufs = [np.zeros(3) for _ in range(M)]
    frame_counts = np.zeros(M, dtype=int)
    prev_phase = np.zeros(M)
    prev_tgradw = np.zeros(M)
    prev_slog = np.full(M, -100.0)
    prev_time = np.full(M, -1.0)
    latest_slog = np.full(M, -100.0)
    latest_fgrad = np.zeros(M)
    latest_fgrad_raw = np.zeros(M)
    phase_out = [np.zeros(N[m]) for m in range(M)]

    i = 0
    while i < len(events):
        t_now = events[i][0]
        batch = []
        while i < len(events) and events[i][0] == t_now:
            batch.append(events[i])
            i += 1

        for (t_ev, m, n) in batch:
            mag_val = abs(s_list[m][n])
            slog_val = math.log(mag_val + np.finfo(float).tiny)
            buf = log_bufs[m]
            buf[0] = buf[1]; buf[1] = buf[2]; buf[2] = slog_val
            frame_counts[m] += 1
            cnt = int(frame_counts[m])
            tfr_m = sqtfr[m] ** 2
            if cnt >= 3:
                fd = (3.0 * buf[2] - 4.0 * buf[1] + buf[0]) / 2.0
            elif cnt >= 2:
                fd = buf[2] - buf[1]
            else:
                fd = 0.0
            latest_slog[m] = slog_val
            latest_fgrad_raw[m] = fd
            latest_fgrad[m] = fd * tfr_m / (2.0 * np.pi)

        # tgrad with time-offset correction for non-uniform hop sizes
        time_curr = np.array([t_now] * M, dtype=float)
        tgrad_curr = _causal_tgrad_tick(
            latest_slog, fc, sqtfr, M, L,
            fgrad_raw=latest_fgrad_raw, time_pos=time_curr, a=a.astype(float))
        tgradw_curr = (tgrad_curr + fc) * np.pi
        fgradw_curr = -latest_fgrad * np.pi

        phase = _heap_phase_tick(
            prev_slog, latest_slog, prev_tgradw, tgradw_curr,
            fgradw_curr, fc, prev_phase, tol, M, prev_time, time_curr
        )

        for (t_ev, m, n) in batch:
            phase_out[m][n] = phase[m]

        prev_phase = phase.copy()
        prev_tgradw = tgradw_curr.copy()
        prev_slog = latest_slog.copy()
        prev_time = time_curr.copy()

    return [np.abs(np.asarray(s_list[m])) * np.exp(1j * phase_out[m]) for m in range(M)]

## Generate Test Signals

In [ ]:
def make_signals(fs, n=10, dur=0.5):
    """Generate diverse synthetic test signals."""
    Ls = int(fs * dur)
    t = np.arange(Ls) / fs
    rng = np.random.default_rng(42)
    signals = []
    for i in range(n):
        sig_type = i % 4
        if sig_type == 0:  # Multi-sinusoidal
            n_sines = rng.integers(3, 10)
            freqs = rng.uniform(80, fs/2 - 200, n_sines)
            amps = rng.uniform(0.1, 1.0, n_sines)
            phases = rng.uniform(0, 2*np.pi, n_sines)
            s = sum(a * np.sin(2*np.pi*f*t + p) for a, f, p in zip(amps, freqs, phases))
        elif sig_type == 1:  # Chirp
            f0 = rng.uniform(100, 500)
            f1 = rng.uniform(2000, 6000)
            s = np.sin(2*np.pi * (f0*t + (f1-f0)/(2*dur)*t**2))
        elif sig_type == 2:  # AM-FM
            fc = rng.uniform(200, 2000)
            fm = rng.uniform(2, 10)
            beta = rng.uniform(100, 500)
            am_freq = rng.uniform(3, 8)
            s = (1 + 0.5*np.sin(2*np.pi*am_freq*t)) * np.sin(
                2*np.pi*fc*t + beta*np.sin(2*np.pi*fm*t))
        else:  # Shaped noise
            noise = rng.standard_normal(Ls)
            freqs_fft = np.fft.rfftfreq(Ls, 1/fs)
            H = np.exp(-((freqs_fft - 500)**2) / (2 * 300**2))
            H += 0.5 * np.exp(-((freqs_fft - 2000)**2) / (2 * 500**2))
            S = np.fft.rfft(noise) * H
            s = np.fft.irfft(S, Ls)
        s = s / (np.max(np.abs(s)) + 1e-10) * 0.9
        signals.append(s)
    return signals, Ls

fs = 16000
signals, Ls = make_signals(fs, n=10, dur=0.5)
print(f'Generated {len(signals)} signals, each {Ls} samples ({Ls/fs:.1f}s)')

## Run Experiments (Table I)

This reproduces Table I from the paper. It compares six methods on the auditory filterbank.

In [ ]:
# Set up filterbank: 134-channel ERB, half-ERB spacing, redmul=16 => ~77x redundancy
redmul = 16.0
spacing = 0.25
g, a, fc_hz, L = audfilters(fs, Ls, redmul=redmul, spacing=spacing)
M = len(g)
a_norm = normalise_a(a, M)
a_int = np.array([int(a_norm[m, 0]) for m in range(M)])
fc_norm_deriv = get_fc_norm(g, L)  # For derivative-filter PGHI
fc_norm_stream = np.array(fc_hz) / fs * 2.0  # For streaming (magnitude-based)
tfr = compute_tfr(g, L)
sqtfr = np.sqrt(np.abs(tfr))
gd = filterbankrealdual(g, a_norm, L)
N = [L // a_int[m] for m in range(M)]
redundancy = sum(N) / L

print(f'Filterbank: M={M}, L={L}, redundancy={redundancy:.1f}x')
print(f'Hop range: [{min(a_int)}, {max(a_int)}]')

In [ ]:
# Run all methods
results = {
    'RTPGHI (batch, deriv. grad.)': {'sc': [], 'sdr': []},
    'Streaming RTPGHI (heap)': {'sc': [], 'sdr': []},
    'Streaming Diff-RTPGHI': {'sc': [], 'sdr': []},
    'fGLA (PGHI init, 100 it.)': {'sc': [], 'sdr': []},
    'fGLA (zero init, 100 it.)': {'sc': [], 'sdr': []},
    'ADMM (100 it.)': {'sc': [], 'sdr': []},
    'RAAR (100 it., beta=0.9)': {'sc': [], 'sdr': []},
    'DM (100 it., beta=0.8)': {'sc': [], 'sdr': []},
    'Zero phase': {'sc': [], 'sdr': []},
}

for i, sig in enumerate(signals):
    sig_padded = np.zeros(L)
    sig_padded[:min(Ls, L)] = sig[:min(Ls, L)]
    c_orig = filterbank(sig_padded, g, a_norm, L=L)
    s_list = [np.abs(np.asarray(ci).ravel()) for ci in c_orig]

    # 1. Batch heap PGHI
    c_bh, _ = batch_heap_pghi(sig_padded, g, a_norm, L, a_int, fc_norm_deriv)
    results['RTPGHI (batch, deriv. grad.)']['sc'].append(sc_roundtrip(c_orig, c_bh, g, a_norm, L))
    results['RTPGHI (batch, deriv. grad.)']['sdr'].append(align_phase_sdr(sig, c_bh, gd, a_norm, L, Ls, 360))

    # 2. Streaming heap PGHI
    c_sh = streaming_heap_pghi(s_list, a_int, fc_norm_stream, sqtfr, L)
    results['Streaming RTPGHI (heap)']['sc'].append(sc_roundtrip(c_orig, c_sh, g, a_norm, L))
    results['Streaming RTPGHI (heap)']['sdr'].append(align_phase_sdr(sig, c_sh, gd, a_norm, L, Ls, 360))

    # 3. Streaming Diff-RTPGHI
    c_sf, _, _, _ = constphase_nonuniform(s_list, a_int, fc_norm_stream, tfr, tol=1e-6)
    results['Streaming Diff-RTPGHI']['sc'].append(sc_roundtrip(c_orig, c_sf, g, a_norm, L))
    results['Streaming Diff-RTPGHI']['sdr'].append(align_phase_sdr(sig, c_sf, gd, a_norm, L, Ls, 360))

    # 4. fGLA from PGHI init
    try:
        c_fg, _, _, _ = gla(c_bh, g, a_norm, L=L, real=True, maxit=100, method='fgla', startphase='input')
        results['fGLA (PGHI init, 100 it.)']['sc'].append(sc_roundtrip(c_orig, c_fg, g, a_norm, L))
        results['fGLA (PGHI init, 100 it.)']['sdr'].append(align_phase_sdr(sig, c_fg, gd, a_norm, L, Ls, 360))
    except:
        results['fGLA (PGHI init, 100 it.)']['sc'].append(float('nan'))
        results['fGLA (PGHI init, 100 it.)']['sdr'].append(float('nan'))

    # 5. fGLA from zero init
    try:
        c_fz_in = [s.astype(complex) for s in s_list]
        c_fz, _, _, _ = gla(c_fz_in, g, a_norm, L=L, real=True, maxit=100, method='fgla', startphase='input')
        results['fGLA (zero init, 100 it.)']['sc'].append(sc_roundtrip(c_orig, c_fz, g, a_norm, L))
        results['fGLA (zero init, 100 it.)']['sdr'].append(align_phase_sdr(sig, c_fz, gd, a_norm, L, Ls, 360))
    except:
        results['fGLA (zero init, 100 it.)']['sc'].append(float('nan'))
        results['fGLA (zero init, 100 it.)']['sdr'].append(float('nan'))

    # 6. ADMM (100 iterations)
    try:
        c_admm, _, _, _ = admm(s_list, g, a_norm, L=L, real=True, maxit=100, startphase='zero')
        results['ADMM (100 it.)']['sc'].append(sc_roundtrip(c_orig, c_admm, g, a_norm, L))
        results['ADMM (100 it.)']['sdr'].append(align_phase_sdr(sig, c_admm, gd, a_norm, L, Ls, 360))
    except:
        results['ADMM (100 it.)']['sc'].append(float('nan'))
        results['ADMM (100 it.)']['sdr'].append(float('nan'))

    # 7. RAAR (100 iterations, beta=0.9)
    try:
        c_raar, _, _, _ = raar(s_list, g, a_norm, L=L, real=True, maxit=100, beta=0.9, startphase='zero')
        results['RAAR (100 it., beta=0.9)']['sc'].append(sc_roundtrip(c_orig, c_raar, g, a_norm, L))
        results['RAAR (100 it., beta=0.9)']['sdr'].append(align_phase_sdr(sig, c_raar, gd, a_norm, L, Ls, 360))
    except:
        results['RAAR (100 it., beta=0.9)']['sc'].append(float('nan'))
        results['RAAR (100 it., beta=0.9)']['sdr'].append(float('nan'))

    # 8. DM (100 iterations, beta=0.8)
    try:
        c_dm, _, _, _ = dm(s_list, g, a_norm, L=L, real=True, maxit=100, beta=0.8, startphase='zero')
        results['DM (100 it., beta=0.8)']['sc'].append(sc_roundtrip(c_orig, c_dm, g, a_norm, L))
        results['DM (100 it., beta=0.8)']['sdr'].append(align_phase_sdr(sig, c_dm, gd, a_norm, L, Ls, 360))
    except:
        results['DM (100 it., beta=0.8)']['sc'].append(float('nan'))
        results['DM (100 it., beta=0.8)']['sdr'].append(float('nan'))

    # 9. Zero phase
    c_zp = [s.astype(complex) for s in s_list]
    results['Zero phase']['sc'].append(sc_roundtrip(c_orig, c_zp, g, a_norm, L))
    results['Zero phase']['sdr'].append(align_phase_sdr(sig, c_zp, gd, a_norm, L, Ls, 360))

    if (i + 1) % 10 == 0:
        print(f'  Processed {i+1}/{len(signals)} signals...')

print('Done!')

In [ ]:
# Display Table I
print('\n' + '='*70)
print('TABLE I: Phase retrieval quality on auditory filterbank')
print('        (mean +/- std over 30 signals, dB)')
print('='*70)
print(f'{"Method":<35s} {"SC (dB)":>14s} {"SDR (dB)":>14s}')
print('-'*70)

for name, data in results.items():
    sc_vals = [v for v in data['sc'] if not np.isnan(v)]
    sdr_vals = [v for v in data['sdr'] if not np.isnan(v)]
    if sc_vals:
        sc_str = f'{np.mean(sc_vals):6.1f} +/- {np.std(sc_vals):4.1f}'
        sdr_str = f'{np.mean(sdr_vals):6.1f} +/- {np.std(sdr_vals):4.1f}'
        marker = '>>>' if 'Diff-RTPGHI' in name else '   '
        print(f'{marker} {name:<32s} {sc_str:>14s} {sdr_str:>14s}')

# Streaming gap
sh = np.array(results['Streaming RTPGHI (heap)']['sdr'])
sf = np.array(results['Streaming Diff-RTPGHI']['sdr'])
gap = sf - sh
print(f'\nStreaming SDR gap (Diff - Heap): {np.mean(gap):+.3f} +/- {np.std(gap):.3f} dB')

sh_sc = np.array(results['Streaming RTPGHI (heap)']['sc'])
sf_sc = np.array(results['Streaming Diff-RTPGHI']['sc'])
gap_sc = sf_sc - sh_sc
print(f'Streaming SC gap (Diff - Heap):  {np.mean(gap_sc):+.3f} +/- {np.std(gap_sc):.3f} dB')

## Generate Figure 1: Gradient Flow Comparison

In [ ]:
# Compute Jacobian for one streaming tick (same as paper Figure 1)
sig_fig = signals[0]
sig_fig_padded = np.zeros(L)
sig_fig_padded[:min(Ls, L)] = sig_fig[:min(Ls, L)]
c_fig = filterbank(sig_fig_padded, g, a_norm, L=L)
s_list_fig = [np.abs(np.asarray(ci).ravel()) for ci in c_fig]

# Build events and process to a middle tick
events = []
for m in range(M):
    N_m = len(s_list_fig[m])
    for n in range(N_m):
        events.append((n * a_int[m], m, n))
events.sort(key=lambda x: (x[0], x[1]))
tick_times = sorted(set(ev[0] for ev in events))
target_tick = tick_times[len(tick_times) // 3]

log_bufs = [np.zeros(3) for _ in range(M)]
frame_counts = np.zeros(M, dtype=int)
prev_phase = np.zeros(M)
prev_tgradw = np.zeros(M)
prev_slog = np.full(M, -100.0)
latest_slog = np.full(M, -100.0)
latest_fgrad = np.zeros(M)
latest_fgrad_raw = np.zeros(M)

idx = 0
while idx < len(events):
    t_now = events[idx][0]
    batch = []
    while idx < len(events) and events[idx][0] == t_now:
        batch.append(events[idx])
        idx += 1
    for (t_ev, m_ev, n_ev) in batch:
        mag_val = abs(s_list_fig[m_ev][n_ev])
        slog_val = math.log(mag_val + np.finfo(float).tiny)
        buf = log_bufs[m_ev]
        buf[0] = buf[1]; buf[1] = buf[2]; buf[2] = slog_val
        frame_counts[m_ev] += 1
        cnt = int(frame_counts[m_ev])
        tfr_m = sqtfr[m_ev] ** 2
        fd = (3*buf[2]-4*buf[1]+buf[0])/2 if cnt >= 3 else (buf[2]-buf[1] if cnt >= 2 else 0)
        latest_slog[m_ev] = slog_val
        latest_fgrad_raw[m_ev] = fd
        latest_fgrad[m_ev] = fd * tfr_m / (2*np.pi)

    time_curr = np.array([t_now]*M, dtype=float)
    tgrad_curr = _causal_tgrad_tick(
        latest_slog, fc_norm_stream, sqtfr, M, L,
        fgrad_raw=latest_fgrad_raw, time_pos=time_curr, a=a_int.astype(float))
    tgradw_curr = (tgrad_curr + fc_norm_stream) * np.pi
    fgradw_curr = -latest_fgrad * np.pi
    time_prev = np.full(M, max(t_now-1, 0))
    phase = _fixed_order_phase_tick(
        prev_slog, latest_slog, prev_tgradw, tgradw_curr,
        fgradw_curr, fc_norm_stream, prev_phase, 1e-6, M, time_prev, time_curr)
    prev_phase = phase.copy()
    prev_tgradw = tgradw_curr.copy()
    prev_slog = latest_slog.copy()
    if t_now >= target_tick:
        break

# Now compute the Jacobian of phase w.r.t. magnitudes at this tick
# Use finite differences
print(f"Computing Jacobian at tick t={t_now} for {M} channels...")
mag_tick = np.array([abs(s_list_fig[m_ev][n_ev]) for (t_ev, m_ev, n_ev) in batch])
eps = 1e-5

def phase_from_mag(mags):
    """Given magnitudes for this tick, compute phase output."""
    slog = np.log(mags + np.finfo(float).tiny)
    # Update latest_slog for this evaluation
    ls = latest_slog.copy()
    lfr = latest_fgrad_raw.copy()
    lf = latest_fgrad.copy()
    for idx_b, (t_ev, m_ev, n_ev) in enumerate(batch):
        ls[m_ev] = slog[idx_b]
        # Recompute fgrad_raw for changed channels
        buf_copy = log_bufs[m_ev].copy()
        buf_copy[2] = slog[idx_b]
        cnt = int(frame_counts[m_ev])
        tfr_m = sqtfr[m_ev] ** 2
        fd = (3*buf_copy[2]-4*buf_copy[1]+buf_copy[0])/2 if cnt >= 3 else (buf_copy[2]-buf_copy[1] if cnt >= 2 else 0)
        lfr[m_ev] = fd
        lf[m_ev] = fd * tfr_m / (2*np.pi)
    tc = np.array([t_now]*M, dtype=float)
    tg = _causal_tgrad_tick(ls, fc_norm_stream, sqtfr, M, L,
                            fgrad_raw=lfr, time_pos=tc, a=a_int.astype(float))
    tgw = (tg + fc_norm_stream) * np.pi
    fgw = -lf * np.pi
    tp = np.full(M, max(t_now-1, 0))
    return _fixed_order_phase_tick(
        prev_slog, ls, prev_tgradw, tgw,
        fgw, fc_norm_stream, prev_phase, 1e-6, M, tp, tc)

phase_base = phase_from_mag(mag_tick)
n_batch = len(batch)
J = np.zeros((M, n_batch))
for j in range(n_batch):
    mag_plus = mag_tick.copy()
    mag_plus[j] += eps
    phase_plus = phase_from_mag(mag_plus)
    J[:, j] = (phase_plus - phase_base) / eps

print(f"Jacobian shape: {J.shape}")
print(f"Non-zero entries: {np.count_nonzero(np.abs(J) > 1e-8)} / {J.size}")

In [ ]:
# Plot Figure 1
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)

mags = np.exp(latest_slog)
mags_norm = mags / (mags.max() + 1e-30)
ax1.bar(range(M), mags_norm, color='steelblue', alpha=0.7, width=0.8)
ax1.set_ylabel('$|s_m|$ (norm.)', fontsize=10)

ax2.plot(range(M), grad_mag_norm, 'o-', color='#d62728', linewidth=1.2,
         markersize=4, label='Diff-RTPGHI')
ax2.axhline(y=0, color='gray', linewidth=0.8, linestyle='--',
            label='RTPGHI (heap): undefined')
ax2.set_xlabel('Channel index $m$', fontsize=10)
ax2.set_ylabel('$||\\partial\\phi/\\partial s_m||$ (norm.)', fontsize=10)
ax2.legend(loc='upper right', fontsize=9)
ax2.set_xlim(-0.5, M - 0.5)

plt.suptitle('Figure 1: Gradient flow comparison at one streaming tick', fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print(f'Non-zero gradients: {np.sum(grad_mag > 1e-8)}/{M} channels')